In [232]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [233]:
os.listdir('/data/aman_singh/acuuracy_check')

['missing_keys_drm.csv',
 'Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'seasonality_all2.csv',
 'chek_nan.csv',
 'missing_keys_drm2.csv',
 'all_combination_ecom_may_pred.csv',
 'QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 'soh_recent_qcom.csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'April-26 Plans.xlsx',
 'QCOM Chain PSKU OTP Output',
 'Heuristics_all_combination_ecom_may_live.xlsx',
 'Norms 202602.csv',
 'duplicates_after_realignment.csv',
 'ALL Channels Accuracy_fva.ipynb',
 "mt_channels Live Run may'26.csv",
 't_thres_df_2.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'qcom_chain_depot_psku_primary_forecast.csv',
 "gt_channels Live Run may'26.csv",
 'seasonality.xlsx',
 'ECOM Chain PSKU Primary_as_on_12_Jan_2026 (1).xlsb',
 'QCOM_NORMS_FINAL_offtake_to_secondary_chain_depot_psku.ipynb',
 'soh_base_may_run.csv',
 'combine_model+missing_forecasts.ipynb',
 'Heuristics_all_combination_ecom_apr_live.xlsx',
 'trend_gt4.csv',
 'handle_missi

In [234]:
base_dir = '/data/aman_singh/acuuracy_check'
input_table = 'TRN_DF_QCOM_OFFTAKE_CHAIN_DEPOT_PSKU'
run_month = '2026-06-30'

In [235]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [236]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/missing_keys_drm.csv',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 '/data/aman_singh/acuuracy_check/seasonality_all2.csv',
 '/data/aman_singh/acuuracy_check/chek_nan.csv',
 '/data/aman_singh/acuuracy_check/missing_keys_drm2.csv',
 '/data/aman_singh/acuuracy_check/all_combination_ecom_may_pred.csv',
 '/data/aman_singh/acuuracy_check/QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 '/data/aman_singh/acuuracy_check/soh_recent_qcom.csv',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/April-26 Plans.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_may_live.xlsx',
 '/data/aman_singh/acuuracy_check/Norms 202602.csv',
 '/data/aman_singh/acuuracy_check/duplicates_after_realignment.csv',
 '/data/aman_singh/acuuracy_check/ALL Channels Accuracy_fva.ipynb',
 "/data/aman_singh/acuuracy_check/mt_channels Live Run may'26.csv",


In [237]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [238]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [239]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-31' and run_month = '{run_month}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,DEPOT,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,IMPUTED,RUN_MONTH
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-06-30
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30


In [240]:
offtake_df.columns = offtake_df.columns.str.lower()

In [241]:
# offtake_df = pd.read_csv('/data/aman_singh/mt_forecast/qcom_depot_psku_data.csv')
# offtake_df = offtake_df[offtake_df['month_date']<='2026-03-31']
# offtake_df

In [242]:
offtake_df.duplicated(
    subset=['platform_name','depot','parent_material_code', 'month_date']).sum()

0

In [243]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [244]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [245]:
offtake_df['key'] = offtake_df[['platform_name','depot','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [246]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [247]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

False

In [248]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [249]:
base_dir

'/data/aman_singh/acuuracy_check'

In [250]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [251]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026.csv


In [252]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [253]:
# forecast_train_till_file_df

In [254]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,vol_in_rum,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.092660,0.048333,0.000671,0.000336,0.001287,0.000671,718288,D112,blinkit,0.145,SAFF GOLD,138865.260689,0.002014,NaN,NaN,0.145,0.002014,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.001315,0.019333,0.000671,0.000336,0.000018,0.000268,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.003047,0.004833,0.000671,0.000336,0.000042,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.004675,0.004833,0.000671,0.000336,0.000065,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.006517,0.004833,0.000000,0.000336,0.000090,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,zepto_D677_810439,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
259198,zepto_D677_810439,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
259199,zepto_D677_810439,2026-08-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
259200,zepto_D677_810439,2026-09-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...


In [255]:
# prophet_file_df.to_csv('Prophet_file_OT_FK_AZ_BB_.csv', index=False)

In [256]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'depot', 'platform_name',
       'vol_in_rum', 'brand_code', 'qtr_ind_rate', 'vol_in_rum_value',
       'pred_best_model', 'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path'],
      dtype='object')

In [257]:
trend_file_df.dtypes

key                          object
month_date                   object
pred_p3m                    float64
pred_p6m                    float64
pred_prophet                float64
pred_rf                     float64
pred_value_p3m              float64
pred_value_p6m              float64
pred_value_prophet          float64
pred_value_rf               float64
parent_material_code          int64
depot                        object
platform_name                object
vol_in_rum                  float64
brand_code                   object
qtr_ind_rate                float64
vol_in_rum_value            float64
pred_best_model             float64
pred_value_best_model       float64
vol_in_rum_treated          float64
vol_in_rum_value_treated    float64
train_till                   object
cov                         float64
run                          object
step                         object
file_path                    object
dtype: object

In [258]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [259]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [260]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [261]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [262]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,vol_in_rum,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.092660,0.048333,0.000671,0.000336,0.001287,0.000671,718288,D112,blinkit,0.145,SAFF GOLD,138865.260689,0.002014,NaN,NaN,0.145,0.002014,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.001315,0.019333,0.000671,0.000336,0.000018,0.000268,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.003047,0.004833,0.000671,0.000336,0.000042,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.004675,0.004833,0.000671,0.000336,0.000065,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.006517,0.004833,0.000000,0.000336,0.000090,0.000067,718288,D112,blinkit,0.000,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,zepto_D677_810439,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M
259198,zepto_D677_810439,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1
259199,zepto_D677_810439,2026-08-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2
259200,zepto_D677_810439,2026-09-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,0.000,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3


In [263]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,vol_in_rum,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
25,blinkit_D112_718288,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,D112,blinkit,0.0,SAFF GOLD,138865.260689,0.0,0.0,0.0,0.0,0.0,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M
26,blinkit_D112_718288,2026-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,D112,blinkit,0.0,SAFF GOLD,138865.260689,0.0,0.0,0.0,0.0,0.0,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1
27,blinkit_D112_718288,2026-08-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,D112,blinkit,0.0,SAFF GOLD,138865.260689,0.0,0.0,0.0,0.0,0.0,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2
28,blinkit_D112_718288,2026-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,D112,blinkit,0.0,SAFF GOLD,138865.260689,0.0,0.0,0.0,0.0,0.0,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
29,blinkit_D112_718288,2026-10-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,D112,blinkit,0.0,SAFF GOLD,138865.260689,0.0,0.0,0.0,0.0,0.0,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,zepto_D677_810439,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,810439,D677,zepto,0.0,SAF-MUSLI,315513.490535,0.0,0.0,0.0,0.0,0.0,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M
259198,zepto_D677_810439,2026-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,810439,D677,zepto,0.0,SAF-MUSLI,315513.490535,0.0,0.0,0.0,0.0,0.0,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1
259199,zepto_D677_810439,2026-08-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,810439,D677,zepto,0.0,SAF-MUSLI,315513.490535,0.0,0.0,0.0,0.0,0.0,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2
259200,zepto_D677_810439,2026-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,810439,D677,zepto,0.0,SAF-MUSLI,315513.490535,0.0,0.0,0.0,0.0,0.0,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3


In [264]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [265]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [266]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4'], dtype=object)

In [267]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [268]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [269]:
trend_file_df['portfolio'].isna().sum()

0

In [270]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [271]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,trend_70_%ile,trend_75_%ile,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,key,y,month_date,brand_code,qtr_ind_rate,vol_in_rum,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,run_month
0,2024-05-31,0.011367,0.073492,0.110628,0.011367,0.011367,0.096379,0.100956,0.102720,0.011367,0.011367,0.011367,0.081293,0.081293,0.081293,0.081293,0.081293,0.081293,0.0,0.0,0.0,0.092660,blinkit_D112_718288,0.145,2024-05-31,SAFF GOLD,138865.260689,0.145,0.002014,0.001287,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
1,2024-06-30,0.010298,-0.018289,0.021391,0.010298,0.010298,0.005674,0.009374,0.011834,0.010298,0.010298,0.010298,-0.008983,-0.008983,-0.008983,-0.008983,-0.008983,-0.008983,0.0,0.0,0.0,0.001315,blinkit_D112_718288,0.000,2024-06-30,SAFF GOLD,138865.260689,0.000,0.000000,0.000018,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
2,2024-07-31,0.009193,-0.015699,0.023804,0.009193,0.009193,0.007322,0.011299,0.013888,0.009193,0.009193,0.009193,-0.006147,-0.006147,-0.006147,-0.006147,-0.006147,-0.006147,0.0,0.0,0.0,0.003047,blinkit_D112_718288,0.000,2024-07-31,SAFF GOLD,138865.260689,0.000,0.000000,0.000042,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
3,2024-08-31,0.008089,-0.014741,0.024127,0.008089,0.008089,0.008798,0.012763,0.014733,0.008089,0.008089,0.008089,-0.003414,-0.003414,-0.003414,-0.003414,-0.003414,-0.003414,0.0,0.0,0.0,0.004675,blinkit_D112_718288,0.000,2024-08-31,SAFF GOLD,138865.260689,0.000,0.000000,0.000065,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
4,2024-09-30,0.007020,-0.013603,0.026860,0.007020,0.007020,0.010185,0.014201,0.016709,0.007020,0.007020,0.007020,-0.000503,-0.000503,-0.000503,-0.000503,-0.000503,-0.000503,0.0,0.0,0.0,0.006517,blinkit_D112_718288,0.000,2024-09-30,SAFF GOLD,138865.260689,0.000,0.000000,0.000090,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,2026-06-30,-0.000411,-0.000603,-0.000603,-0.000411,-0.000411,-0.000603,-0.000603,-0.000603,-0.000411,-0.000411,-0.000411,-0.000192,-0.000192,-0.000192,-0.000192,-0.000192,-0.000192,0.0,0.0,0.0,0.000000,zepto_D677_810439,0.000,2026-06-30,SAF-MUSLI,315513.490535,0.000,0.000000,0.000000,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
259198,2026-07-31,-0.000469,-0.000693,-0.000693,-0.000470,-0.000469,-0.000693,-0.000693,-0.000693,-0.000469,-0.000469,-0.000469,-0.000224,-0.000224,-0.000224,-0.000224,-0.000224,-0.000224,0.0,0.0,0.0,0.000000,zepto_D677_810439,0.000,2026-07-31,SAF-MUSLI,315513.490535,0.000,0.000000,0.000000,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
259199,2026-08-31,-0.000528,-0.000559,-0.000557,-0.000528,-0.000527,-0.000558,-0.000558,-0.000558,-0.000528,-0.000527,-0.000527,-0.000031,-0.000031,-0.000031,-0.000031,-0.000031,-0.000031,0.0,0.0,0.0,0.000000,zepto_D677_810439,0.000,2026-08-31,SAF-MUSLI,315513.490535,0.000,0.000000,0.000000,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
259200,2026-09-30,-0.000584,-0.000953,-0.000950,-0.000586,-0.000583,-0.000951,-0.000951,-0.000951,-0.000584,-0.000584,-0.000584,-0.000367,-0.000367,-0.000367,-0.000367,-0.000367,-0.000367,0.0,0.0,0.0,0.000000,zepto_D677_810439,0.000,2

In [272]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile', 'yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [273]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [274]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [275]:
offtake_df.head()

,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-06-30
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30


In [276]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [277]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
# offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])

In [278]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key','month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [279]:
trend_file_df.select_dtypes('number').isna().sum()

pred_p3m                         0
pred_p6m                         0
pred_prophet                     0
pred_rf                          0
pred_value_p3m                   0
pred_value_p6m                   0
pred_value_prophet               0
pred_value_rf                    0
parent_material_code             0
qtr_ind_rate                     0
vol_in_rum_value                 0
pred_best_model             219037
pred_value_best_model       219037
vol_in_rum_treated               0
vol_in_rum_value_treated         0
cov                              0
pred_prophet_70%ile              0
pred_prophet_60%ile              0
vol_in_rum                       0
dtype: int64

In [280]:
trend_file_df.select_dtypes('number').min().round()

pred_p3m                         0.0
pred_p6m                         0.0
pred_prophet                     0.0
pred_rf                          0.0
pred_value_p3m                   0.0
pred_value_p6m                   0.0
pred_value_prophet               0.0
pred_value_rf                    0.0
parent_material_code        715101.0
qtr_ind_rate                   100.0
vol_in_rum_value                 0.0
pred_best_model                  0.0
pred_value_best_model            0.0
vol_in_rum_treated               0.0
vol_in_rum_value_treated         0.0
cov                              0.0
pred_prophet_70%ile           -359.0
pred_prophet_60%ile           -366.0
vol_in_rum                       0.0
dtype: float64

In [281]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [282]:
for col in [ 'pred_best_model', 'pred_value_best_model', 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [283]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.092660,0.048333,0.000671,0.000336,0.001287,0.000671,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,NaN,NaN,0.145,0.002014,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.100956,0.096379,0.145
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.001315,0.019333,0.000671,0.000336,0.000018,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.009374,0.005674,0.000
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.003047,0.004833,0.000671,0.000336,0.000042,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.011299,0.007322,0.000
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.004675,0.004833,0.000671,0.000336,0.000065,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.012763,0.008798,0.000
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.006517,0.004833,0.000000,0.000336,0.000090,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.014201,0.010185,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,zepto_D677_810439,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M,Foods,0.000000,0.000000,0.000
259198,zepto_D677_810439,2026-07-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1,Foods,0.000000,0.000000,0.000
259199,zepto_D677_810439,2026-08-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2,Foods,0.000000,0.000000,0.000
259200,zepto_D677_810439,2026-09-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810439,D677,zepto,SAF-MUSLI,315513.490535,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,3.605551,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,Foods,0.000000,0.000000,0.000


In [284]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [285]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [286]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [287]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [288]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [289]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [290]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [291]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [292]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [293]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [294]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [295]:
# trend_file_df.to_csv('collate_check.csv', index=False)

In [296]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [297]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [298]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'pred_value_best_model',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [299]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [300]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
1406,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,NaN,NaN,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000024,1.618828e-05
1407,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000e+00
1408,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000006,1.131882e-07
1409,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,0.000025,NaN,NaN,NaN,0.000026,2.037054e-05
1410,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,-100.0,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,0.000029,2.138962e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165175,swiggy_D530_719192,2026-06-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,6.164414,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00
165176,swiggy_D530_719192,2026-07-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,6.164414,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00
165177,swiggy_D530_719192,2026-08-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,6.164414,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100

In [301]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [302]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [303]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [304]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'depot', 'platform_name',
       'brand_code', 'qtr_ind_rate', 'vol_in_rum_value', 'pred_best_model',
       'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
       'OT_Value_in

In [305]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [306]:
trend_file_df.reset_index(drop=True, inplace=True)

In [307]:
trend_file_df.shape

(259202, 56)

In [308]:
trend_file_df['key'].nunique()

8033

In [309]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [310]:
brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
brand_class['Channel'] = brand_class['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
brand_class.columns = brand_class.columns.str.lower()
brand_class = brand_class[brand_class['channel'] == 'QCOM']
brand_class = brand_class[['brand','final class']]

brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [311]:
# trend_file_df.drop(columns = ['final class'], inplace = True)

In [312]:
brand_class

,brand_code,class
845,SAFF GOLD,A
846,PCNO(R),A
847,SAFF ACTV,A
848,SFOATS-FL,A
849,SAFF OATS,A
...,...,...
991,PADV-HRAD,C
992,PADVJAS-F,C
993,SFFT_VNGR,C
994,PURSNS_GM,C


In [313]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_class, 
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge

In [314]:
trend_file_df['class'].isna().sum()

1181

In [315]:
trend_file_df['class'].unique()

array(['C', 'NPD', nan, 'A', 'B'], dtype=object)

In [316]:
trend_file_df['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [317]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ].to_csv('Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv', index=False)

missing combinations

In [318]:
model_file = trend_file_df.copy()

In [319]:
model_file['key'].nunique()

8033

In [320]:
run_month

Timestamp('2026-06-30 00:00:00')

In [321]:
data_query = f"""select * from {input_table} where month_date >= '2023-01-31' and run_month = '2026-06-30' """
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,DEPOT,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,IMPUTED,RUN_MONTH
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-06-30
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30


In [322]:
qcom_df.columns = qcom_df.columns.str.lower()

In [323]:
qcom_df['key'] = qcom_df[['platform_name', 'depot','parent_material_code']].astype(str).agg('_'.join, axis=1)

In [324]:
qcom_df

,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-06-30
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-06-30
...,...,...,...,...,...,...,...,...,...
350377,2026-11-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-06-30
350378,2026-12-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-06-30
350379,2027-01-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-06-30
350380,2027-02-28,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-06-30


In [325]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [326]:
qcom_df.shape

(350382, 9)

In [327]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key','run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month,LY
1611,2025-10-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0055,0,2026-06-30,NaN
1612,2025-11-30,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0185,0,2026-06-30,NaN
1613,2025-12-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0250,0,2026-06-30,NaN
1614,2026-01-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0290,0,2026-06-30,NaN
1615,2026-02-28,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0275,0,2026-06-30,NaN
...,...,...,...,...,...,...,...,...,...,...
347304,2026-11-30,zepto_D676_810439,zepto,D676,810439,SAF-MUSLI,0.0000,1,2026-06-30,NaN
347305,2026-12-31,zepto_D676_810439,zepto,D676,810439,SAF-MUSLI,0.0000,1,2026-06-30,NaN
347306,2027-01-31,zepto_D676_810439,zepto,D676,810439,SAF-MUSLI,0.0000,1,2026-06-30,NaN
347307,2027-02-28,zepto_D676_810439,zepto,D676,810439,SAF-MUSLI,0.0000,1,2026-06-30,NaN


In [328]:
missing_df['key'].nunique()

2958

In [329]:
trend_file_df['key'].nunique()

8033

In [330]:
missing_df.isnull().sum()

month_date                  0
key                         0
platform_name               0
depot                       0
parent_material_code        0
brand_code                  0
vol_in_rum                  0
imputed                     0
run_month                   0
LY                      51015
dtype: int64

In [331]:
missing_df['key'].nunique()

2958

In [332]:

missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'depot','parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,depot,parent_material_code,brand_code,vol_in_rum
1611,blinkit_D112_718592,2026-06-30,2025-10-31,blinkit,D112,718592,SFOATS-FL,0.0055
1612,blinkit_D112_718592,2026-06-30,2025-11-30,blinkit,D112,718592,SFOATS-FL,0.0185
1613,blinkit_D112_718592,2026-06-30,2025-12-31,blinkit,D112,718592,SFOATS-FL,0.0250
1614,blinkit_D112_718592,2026-06-30,2026-01-31,blinkit,D112,718592,SFOATS-FL,0.0290
1615,blinkit_D112_718592,2026-06-30,2026-02-28,blinkit,D112,718592,SFOATS-FL,0.0275
...,...,...,...,...,...,...,...,...
347304,zepto_D676_810439,2026-06-30,2026-11-30,zepto,D676,810439,SAF-MUSLI,0.0000
347305,zepto_D676_810439,2026-06-30,2026-12-31,zepto,D676,810439,SAF-MUSLI,0.0000
347306,zepto_D676_810439,2026-06-30,2027-01-31,zepto,D676,810439,SAF-MUSLI,0.0000
347307,zepto_D676_810439,2026-06-30,2027-02-28,zepto,D676,810439,SAF-MUSLI,0.0000


In [333]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()




Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,TRU_PDRFR,850.57000
1,2027-03-31,TRU_OATS,177.07000
2,2027-03-31,TRU_QUINO,204.75000
3,2027-03-31,TRU_RAW,453.44000
4,2027-03-31,NHR_NHO_E,266.57953


In [334]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [335]:
missing_df

,key,run_month,month_date,platform_name,depot,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,blinkit_D112_718592,2026-06-30,2025-10-31,blinkit,D112,718592,SFOATS-FL,0.0055,292663.137458
1,blinkit_D112_718592,2026-06-30,2025-11-30,blinkit,D112,718592,SFOATS-FL,0.0185,292663.137458
2,blinkit_D112_718592,2026-06-30,2025-12-31,blinkit,D112,718592,SFOATS-FL,0.0250,292663.137458
3,blinkit_D112_718592,2026-06-30,2026-01-31,blinkit,D112,718592,SFOATS-FL,0.0290,292663.137458
4,blinkit_D112_718592,2026-06-30,2026-02-28,blinkit,D112,718592,SFOATS-FL,0.0275,292663.137458
...,...,...,...,...,...,...,...,...,...
51010,zepto_D676_810439,2026-06-30,2026-11-30,zepto,D676,810439,SAF-MUSLI,0.0000,315513.490535
51011,zepto_D676_810439,2026-06-30,2026-12-31,zepto,D676,810439,SAF-MUSLI,0.0000,315513.490535
51012,zepto_D676_810439,2026-06-30,2027-01-31,zepto,D676,810439,SAF-MUSLI,0.0000,315513.490535
51013,zepto_D676_810439,2026-06-30,2027-02-28,zepto,D676,810439,SAF-MUSLI,0.0000,315513.490535


In [336]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)




In [337]:

offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()
missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_class, 
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)
del len_before_merge
missing_df['class'].isna().sum()
missing_df['class'].unique()

LY P3M_value


array(['C', 'NPD', 'A', 'B'], dtype=object)

In [338]:
pd.set_option('display.max_columns', None)

In [339]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [340]:
model_file

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,NaN,NaN,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000024,1.618828e-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,0.000075,NaN,NaN,C
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000006,1.131882e-07,NaN,NaN,NaN,NaN,0.000000,0.000075,NaN,C
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,0.000025,NaN,NaN,NaN,0.000026,2.037054e-05,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000075,C
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,-100.0,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,0.000029,2.138962e-05,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259197,swiggy_D530_719192,2026-06-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,6.164414,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,NaN
259198,swiggy_D530_719192,2026-07-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,6.164414,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00

In [341]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,NaN,NaN,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000024,1.618828e-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,0.000075,NaN,NaN,C,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000006,1.131882e-07,NaN,NaN,NaN,NaN,0.000000,0.000075,NaN,C,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,0.000025,NaN,NaN,NaN,0.000026,2.037054e-05,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000075,C,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,-100.000000,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0.000029,2.138962e-05,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51010,zepto_D673_810125,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,NaN,NaN,0.0,0.906667,NaN,NaN,NaN,NaN,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000178,0.000260,0.000027,NPD,1
51011,zepto_D673_810125,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-06-30,M+6,Male Grooming,NaN,NaN,0.0,0.906667,NaN,NaN,NaN,NaN,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000178,0.000260,0.000027,NPD,1
51012,zepto_D673_810125,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D673,zepto,SW_SGPRF,1712.60

In [342]:
# final_df = trend_file_df.copy()

In [343]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000024,1.618828e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000075,0.000000,0.000000,C,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000006,1.131882e-07,0.00,0.0,0.000000,0.0,0.000000,0.000075,0.000000,C,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.500000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.500000,0.000025,0.0,0.0,0.0,0.000026,2.037054e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000075,C,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,-100.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000029,2.138962e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51010,zepto_D673_810125,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,0.906667,0.0,0.0,0.0,0.0,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000178,0.000260,0.000027,NPD,1
51011,zepto_D673_810125,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.00

In [344]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000024,1.618828e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000075,0.000000,0.000000,C,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000006,1.131882e-07,0.00,0.0,0.000000,0.0,0.000000,0.000075,0.000000,C,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.500000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.500000,0.000025,0.0,0.0,0.0,0.000026,2.037054e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000075,C,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,-100.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000029,2.138962e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51010,zepto_D673_810125,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,0.906667,0.0,0.0,0.0,0.0,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000178,0.000260,0.000027,NPD,1
51011,zepto_D673_810125,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.00

In [345]:
# final_df[
#     # (final_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (final_df['month_date'] > '2024-06-30') &
#     (final_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('all_combinations_QCOM_chain_PSKU_Offtakes_DEC2.csv', index=False)

In [346]:
# final_df.to_csv('heuristic_data_preprocessed_qcom_jan.csv')

In [347]:
# import pandas as pd
# final_df = pd.read_csv('/data/aman_singh/acuuracy_check/heuristic_data_preprocessed_qcom_cp_live_jan.csv')
# final_df

In [348]:
final_df[(final_df['M month'].notna())]#['key'].nunique()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
25,blinkit_D112_718589,2026-06-30,2.1,2.1,2.605122,1.831482,0.000104,0.000104,0.000129,0.000091,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,2.605122,0.000129,0.0,0.0,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M,Hair Oils,2.842577,2.725673,0.0,2.100000,2.1,0.0,0.00,0.0,2.100000,1.750000,75.000000,-14.285714,-33.333333,1.0,2.100000,0.000104,0.000104,0.0,0.000000,0.000141,0.000135,0.60,0.0,0.000030,0.0,0.000164,0.000104,0.000045,C,0
26,blinkit_D112_718589,2026-07-31,2.1,2.1,2.941165,1.718574,0.000104,0.000104,0.000146,0.000085,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,2.941165,0.000146,0.0,0.0,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1,Hair Oils,3.221662,3.085605,0.0,2.100000,2.1,0.0,0.10,0.2,2.100000,1.750000,75.000000,-14.285714,-33.333333,1.0,2.100000,0.000104,0.000104,0.0,0.000005,0.000160,0.000153,1.20,0.0,0.000060,0.0,0.000164,0.000104,0.000045,C,0
27,blinkit_D112_718589,2026-08-31,2.1,2.1,3.467539,2.077455,0.000104,0.000104,0.000172,0.000103,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,3.467539,0.000172,0.0,0.0,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2,Hair Oils,3.726857,3.589988,0.0,2.100000,2.1,0.0,0.30,0.6,2.100000,1.750000,75.000000,-14.285714,-33.333333,1.0,2.100000,0.000104,0.000104,0.0,0.000015,0.000185,0.000178,2.10,0.0,0.000104,0.0,0.000164,0.000104,0.000045,C,0
28,blinkit_D112_718589,2026-09-30,2.1,2.1,3.368910,2.326143,0.000104,0.000104,0.000167,0.000116,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,3.368910,0.000167,0.0,0.0,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,Hair Oils,3.684226,3.499214,0.0,2.100000,2.1,0.0,0.65,1.3,2.100000,1.750000,75.000000,-14.285714,-33.333333,1.0,2.100000,0.000104,0.000104,0.0,0.000032,0.000183,0.000174,2.10,0.0,0.000104,0.0,0.000164,0.000104,0.000045,C,0
29,blinkit_D112_718589,2026-10-31,2.1,2.1,3.108451,2.213672,0.000104,0.000104,0.000154,0.000110,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,3.108451,0.000154,0.0,0.0,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,Hair Oils,3.352689,3.217803,0.0,2.100000,2.1,0.0,1.00,1.8,2.100000,1.750000,75.000000,-14.285714,-33.333333,1.0,2.100000,0.000104,0.000104,0.0,0.000050,0.000167,0.000160,1.80,0.0,0.000089,0.0,0.000164,0.000104,0.000045,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51009,zepto_D673_810125,2026-10-31,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.0,0.000000,0.000000,0.0,0.0,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+4,Male Grooming,0.000000,0.000000,0.0,0.906667,0.0,0.0,0.00,0.0,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,0.000000,0.0,0.000000,0.000000,0.000000,0.00,0.0,0.000000,0.0,0.000178,0.000260,0.000027,NPD,1
51010,zepto_D673_810125,2026-11-30,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D6

In [349]:
final_df[final_df['month_date'] == '2026-05-31']['vol_in_rum_value'].sum()

30.927819258546695

Heuristic new approac

In [350]:
# pip install pymannkendall

In [351]:
# final_df.to_csv('brek2.csv', index=False)

In [352]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "depot","parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [353]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,blinkit,D112,718288,2026-06-30,0,0,0
1,blinkit,D112,718310,2026-06-30,-1,0,0
2,blinkit,D112,718312,2026-06-30,1,1,1
3,blinkit,D112,718317,2026-06-30,-1,0,0
4,blinkit,D112,718319,2026-06-30,-1,0,0
...,...,...,...,...,...,...,...
10986,zepto,D677,810372,2026-06-30,0,0,0
10987,zepto,D677,810405,2026-06-30,0,0,0
10988,zepto,D677,810406,2026-06-30,-1,0,0
10989,zepto,D677,810407,2026-06-30,0,0,0


In [354]:
trend_df[trend_df['final_trend'] == 1]

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
2,blinkit,D112,718312,2026-06-30,1,1,1
13,blinkit,D112,718371,2026-06-30,1,1,1
15,blinkit,D112,718398,2026-06-30,1,1,1
20,blinkit,D112,718461,2026-06-30,1,1,1
21,blinkit,D112,718464,2026-06-30,1,1,1
...,...,...,...,...,...,...,...
10667,zepto,D674,730277,2026-06-30,1,1,1
10717,zepto,D674,809817,2026-06-30,1,1,1
10718,zepto,D674,809818,2026-06-30,1,1,1
10725,zepto,D674,809951,2026-06-30,1,1,1


## detect seasonality

In [355]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,blinkit,ADV-AHO-R,2026-06-30,0
1,blinkit,BIO OILS,2026-06-30,0
2,blinkit,CO_SO_PCP,2026-06-30,0
3,blinkit,CO_SO_VCN,2026-06-30,0
4,blinkit,H&C,2026-06-30,0
...,...,...,...,...
237,zepto,SW HRGEL,2026-06-30,0
238,zepto,SW HSPRY,2026-06-30,0
239,zepto,SW STLDEO,2026-06-30,0
240,zepto,SW_HR_WAX,2026-06-30,0


In [356]:
seasonality_df[seasonality_df['seasonality_flag'] == 1]

,platform_name,brand_code,run_month,seasonality_flag
5,blinkit,H&C DFOIL,2026-06-30,1
59,blinkit,SAF_MAYO,2026-06-30,1
70,blinkit,SF_MNMKHN,2026-06-30,1
72,blinkit,SF_SOYBRJ,2026-06-30,1
75,blinkit,SW NOGAS,2026-06-30,1
140,swiggy,SAF_FT_MR,2026-06-30,1
159,swiggy,SF_SOYBRJ,2026-06-30,1
169,zepto,CO_SO_PCP,2026-06-30,1
173,zepto,JATAA_HO,2026-06-30,1
177,zepto,LIVON 2.0,2026-06-30,1


In [357]:
# seasonality_df.to_csv('seasonality_qcom.csv')

In [358]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "depot","parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,depot,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,D112,718288,2026-06-30,0.000000,0.000000,0.000000,0.000000
1,blinkit,D112,718310,2026-06-30,0.000000,0.004954,0.000934,0.001340
2,blinkit,D112,718312,2026-06-30,0.008525,0.050700,0.025395,0.008435
3,blinkit,D112,718317,2026-06-30,0.000000,0.000000,0.000000,0.000000
4,blinkit,D112,718319,2026-06-30,0.000000,0.000000,0.000000,0.000000


In [359]:
final_df[final_df['key'] == 'blinkit_D112_718317']

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
17218,blinkit_D112_718317,2023-01-31,0.900000,4.766667,3.062103,3.791643,0.000035,1.849831e-04,1.188330e-04,1.471447e-04,718317,D112,blinkit,H&C,388.076436,0.000047,0.000000,0.000000e+00,1.2,0.000047,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,4.303118,3.667845,1.2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000167,0.000142,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,C,0
17219,blinkit_D112_718317,2023-02-28,0.900000,4.766667,2.988898,3.363310,0.000035,1.849831e-04,1.159921e-04,1.305221e-04,718317,D112,blinkit,H&C,388.076436,0.000043,0.000000,0.000000e+00,1.1,0.000043,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,4.252172,3.658546,1.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000165,0.000142,0.0,0.0,0.000000,0.000000,0.000047,0.000000,0.000000,C,0
17220,blinkit_D112_718317,2023-03-31,0.900000,4.766667,2.734864,3.165810,0.000035,1.849831e-04,1.061336e-04,1.228576e-04,718317,D112,blinkit,H&C,388.076436,0.000016,0.000000,0.000000e+00,0.4,0.000016,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,4.020145,3.363739,0.4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000156,0.000131,0.0,0.0,0.000000,0.000000,0.000043,0.000047,0.000000,C,0
17221,blinkit_D112_718317,2023-04-30,0.900000,4.766667,5.053947,7.911143,0.000035,1.849831e-04,1.961318e-04,3.070128e-04,718317,D112,blinkit,H&C,388.076436,0.000407,0.000000,0.000000e+00,10.5,0.000407,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,6.135255,5.474826,10.5,0.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.900000,0.000035,0.000000e+00,0.000000,0.000000e+00,0.000238,0.000212,0.0,0.0,0.000000,0.000000,0.000016,0.000043,0.000047,C,0
17222,blinkit_D112_718317,2023-05-31,4.000000,4.766667,5.061839,6.945500,0.000155,1.849831e-04,1.964380e-04,2.695385e-04,718317,D112,blinkit,H&C,388.076436,0.000427,0.000000,0.000000e+00,11.0,0.000427,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,6.320910,5.642212,11.0,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,344.444444,0.000000,0.000000,0.0,4.000000,0.000155,0.000000e+00,0.000000,0.000000e+00,0.000245,0.000219,0.0,0.0,0.000000,0.000000,0.000407,0.000016,0.000043,C,0
17223,blinkit_D112_718317,2023-06-30,7.300000,4.766667,3.035071,4.898333,0.000283,1.849831e-04,1.177839e-04,1.900928e-04,718317,D112,blinkit,H&C,388.076436,0.000171,0.000000,0.000000e+00,4.4,0.000171,2026-05-31,2.413462,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,4.388820,3.801981,4.4,7.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,82.500000,344.444444,0.000000,0.0,7.300000,

In [360]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'depot','parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,D112,718288,2026-06-30,0,0,0,0.000000,0.000000,0.000000,0.000000
1,blinkit,D112,718310,2026-06-30,-1,0,0,0.000000,0.004954,0.000934,0.001340
2,blinkit,D112,718312,2026-06-30,1,1,1,0.008525,0.050700,0.025395,0.008435
3,blinkit,D112,718317,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
4,blinkit,D112,718319,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
10986,zepto,D677,810372,2026-06-30,0,0,0,0.000000,0.000000,0.000000,0.000000
10987,zepto,D677,810405,2026-06-30,0,0,0,0.000000,0.000000,0.000000,0.000000
10988,zepto,D677,810406,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
10989,zepto,D677,810407,2026-06-30,0,0,0,0.000000,0.000000,0.000000,0.000000


In [361]:
# trend_df.to_csv('t_thres_df_qcom.csv')

In [362]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000024,1.618828e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000075,0.000000,0.000000,C,0,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000006,1.131882e-07,0.00,0.0,0.000000,0.0,0.000000,0.000075,0.000000,C,0,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.500000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.500000,0.000025,0.0,0.0,0.0,0.000026,2.037054e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000075,C,0,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,-100.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000029,2.138962e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310212,zepto_D673_810125,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,0.906667,0.0,0.0,0.0,0.0,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000178,0.000260,0.000027,NPD,1,0
310213,zepto_D673_810125,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,17

In [363]:
trend_df.columns

Index(['platform_name', 'depot', 'parent_material_code', 'run_month',
       'trend_flag', 'p3m_slope_flag', 'final_trend', 'lower_threshold',
       'upper_threshold', 'mean_value', 'std_value'],
      dtype='object')

In [364]:
final_df = final_df.merge(trend_df[['platform_name', 'depot','parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "depot","parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.193725,0.943558,0.000025,0.000012,0.000010,0.000047,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.474310,0.325832,1.5,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000024,1.618828e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0,1,0.000008,0.000221
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,0.840701,0.000025,0.000012,0.000000,0.000042,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.000075,0.000000,0.000000,C,0,0,1,0.000008,0.000221
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.000000,0.327961,0.000025,0.000012,0.000000,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.125969,0.002278,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000006,1.131882e-07,0.00,0.0,0.000000,0.0,0.000000,0.000075,0.000000,C,0,0,1,0.000008,0.000221
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.244806,0.327961,0.000025,0.000012,0.000012,0.000016,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.531383,0.410012,0.0,0.500000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.500000,0.000025,0.0,0.0,0.0,0.000026,2.037054e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000075,C,0,0,1,0.000008,0.000221
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.279075,0.423717,0.000000,0.000012,0.000014,0.000021,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.169268,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,0.580217,0.430523,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,-100.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000029,2.138962e-05,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0,1,0.000008,0.000221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310212,zepto_D673_810125,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125,D673,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,0.906667,0.0,0.0,0.0,0.0,0.746667,0.553333,21.428571,107.407407,28.571429,3.0,0.826667,0.000155,0.0,0.0,0.0,0.000000,0.000000e+00,0.00,0.0,0.000000,0.0,0.00017

In [ ]:
# final_df[final_df['skipped'] == 1]['lower_threshold'].sum()

6.408238205290727

In [366]:
final_df = final_df.sort_values(['key', 'run_month','month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(10)
)


In [126]:
# final_df['stat_bias'] = final_df['Stat Error']/final_df['Actuals Val']
# final_df = final_df.fillna(0)

# import numpy as np
# import pandas as pd

# final_df['stat_bias'] = (
#     final_df['stat_bias']
#     .replace([np.inf, -np.inf], 0)
#     .fillna(0)
# )

# bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
# labels = [
#     '< -15%',
#     '-15% to -10%',
#     '-10% to -5%',
#     '-5% to 0%',
#     '0% to 5%',
#     '5% to 10%',
#     '10% to 15%',
#     '> 15%'
# ]

# final_df['stat_bias_bucket'] = pd.cut(
#     final_df['stat_bias'],
#     bins=bins,
#     labels=labels,
#     right=False  
# )


In [ ]:
# final_df[(final_df['month_date'] == '2026-03-31') & (final_df['run_month'] == '2026-01-31')]['P3M_value'].sum()

26.37755518302953

In [368]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [369]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_D112_718288,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
1,blinkit_D112_718310,2026-06-30,0.000000,0.010500,0.00000,0.000367,0.025500,0.025500,0.000891,0.000891
2,blinkit_D112_718312,2026-06-30,0.992333,0.758833,0.03466,0.026504,0.965667,0.787333,0.033728,0.027500
3,blinkit_D112_718317,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.033333,0.000000,0.000001
4,blinkit_D112_718319,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
10986,zepto_D677_810372,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.050000,0.050000,0.000009,0.000009
10987,zepto_D677_810405,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.015000,0.000000,0.000003
10988,zepto_D677_810406,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.012000,0.000000,0.000002
10989,zepto_D677_810407,2026-06-30,0.000000,0.000000,0.00000,0.000000,0.000000,0.007500,0.000000,0.000001


In [370]:
final_df.shape

(310217, 71)

In [371]:
# adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key'],
    how="left"
)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.092660,0.048333,0.000671,0.000336,0.001287,0.000671,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,0.0,0.0,0.145,0.002014,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.100956,0.096379,0.145,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.001402,0.001338,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,5,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.001315,0.019333,0.000671,0.000336,0.000018,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.009374,0.005674,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000130,0.000079,0.0,0.0,0.0,0.0,0.002014,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,6,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.003047,0.004833,0.000671,0.000336,0.000042,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.011299,0.007322,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000157,0.000102,0.0,0.0,0.0,0.0,0.000000,0.002014,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.004675,0.004833,0.000671,0.000336,0.000065,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.012763,0.008798,0.000,0.048333,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.048333,0.000671,0.0,0.0,0.0,0.000177,0.000122,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002014,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.006517,0.004833,0.000000,0.000336,0.000090,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.014201,0.010185,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,-100.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000197,0.000141,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,9,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.

In [372]:
all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
def detect_month_anomaly(df, brand_code, month_num, threshold=0.25, months_window=3):
    """
    Detect if a specific month's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value'
    - brand_code: filter by this brand code
    - month_num: month to check (6, 7, 8, 9)
    - threshold: 25% difference threshold
    - months_window: number of months before and after to compare
    """
    
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]
    
    anomalies = []
    
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        month_data = df_year[df_year['month'] == month_num]
        if month_data.empty:
            continue
        
        month_value = month_data['vol_in_rum_value'].iloc[0]
        
        past_months = [(month_num - i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        next_months = [(month_num + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
        past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        comparison_values = pd.concat([past_m])
        
        if comparison_values.empty:
            continue
        
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (month_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'month': month_num,
            'year': year,
            'month_value': month_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
        })
    
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Check months 6, 7, 8, 9
brands = all_brand['brand_code'].unique()
results = []

for month in [7, 8, 9,10]:
    for brand in brands:
        df_result, repeats = detect_month_anomaly(all_brand, brand, month)
        if df_result is not None and not df_result.empty:
            df_result['pattern_repeats'] = repeats
            results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

    brand_code  month  year  month_value  num_months_compared  \
33   LVN_SRSNS      7  2025     0.000377                    3   
34   LVN_SRSNS      7  2026     0.000000                    3   
53   PABABY_GM      7  2025     0.039688                    3   
54   PABABY_GM      7  2026     0.000000                    3   
64   PADV_SMPN      7  2025     0.000733                    2   
65   PADV_SMPN      7  2026     0.000000                    3   
75   PA_EXT_ML      7  2025     0.085042                    3   
76   PA_EXT_ML      7  2026     0.000000                    3   
94    P_EN_ALM      7  2025     0.013483                    3   
95    P_EN_ALM      7  2026     0.000000                    3   
127  SAF_CDPRS      7  2025     0.214136                    1   
128  SAF_CDPRS      7  2026     0.000000                    3   
133  SAF_FT_MR      7  2024     0.000000                    3   
134  SAF_FT_MR      7  2025     0.000000                    3   
164  SF_IM_CHY      7  20

In [ ]:
# mnth = 6
# all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
# def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
#     """
#     Detect if April's vol_in_rum_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'vol_in_rum_value', 'run_month'
#     - brand_code: filter by this brand code
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after April to compare
    
#     """
    
#     # Filter for brand and sort by month_date
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     # Extract year and month
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     # Get unique years (excluding current year if incomplete)
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
#     anomalies = []
    
#     # Check each past year's April
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         # Get April data (month == 4)
#         april_data = df_year[df_year['month'] == mnth]
#         if april_data.empty:
#             continue
        
#         april_value = april_data['vol_in_rum_value'].iloc[0]
#         april_month = mnth
        
#         # Dynamically calculate past and next months
#         past_months = [(april_month - i - 1) % 12 + 1 for i in range(1,months_window+1)]
#         print(past_months)
#         next_months = [(april_month + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
#         print(next_months)
        
#         # Get past and next months values
#         past_3m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
#         next_3m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
#         # Combine all comparison months
#         comparison_values = pd.concat([past_3m, next_3m])
        
#         if comparison_values.empty:
#             continue
        
#         # Calculate mean of comparison months
#         #mean_value = comparison_values.mean()
        
#         # Calculate percentage difference
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (april_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         # April is anomalous if it's >25% different from ALL comparison months
#         # AND all differences have the same sign (all positive or all negative)
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'year': year,
#             'april_value': april_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
#         })
    
#     # Check if pattern repeats in both years
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Usage: Apply to each brand code
# brands = all_brand['brand_code'].unique()
# results = []

# for brand in brands:
#     df_result, repeats = detect_april_anomaly(all_brand, brand)
#     if df_result is not None and not df_result.empty:
#         df_result['pattern_repeats'] = repeats
#         results.append(df_result)


# anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]
[4, 3, 2]
[6, 7, 8]


In [134]:
final_df.shape

(29328, 79)

In [373]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code','month'])[['brand_code', 'month','direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(final_brands[['brand_code', 'month','month_different']], on = ['brand_code','month'], how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.092660,0.048333,0.000671,0.000336,0.001287,0.000671,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,0.0,0.0,0.145,0.002014,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.100956,0.096379,0.145,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.001402,0.001338,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,5,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.001315,0.019333,0.000671,0.000336,0.000018,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.009374,0.005674,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000130,0.000079,0.0,0.0,0.0,0.0,0.002014,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,6,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.003047,0.004833,0.000671,0.000336,0.000042,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.011299,0.007322,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000157,0.000102,0.0,0.0,0.0,0.0,0.000000,0.002014,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.004675,0.004833,0.000671,0.000336,0.000065,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.012763,0.008798,0.000,0.048333,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.048333,0.000671,0.0,0.0,0.0,0.000177,0.000122,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002014,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.006517,0.004833,0.000000,0.000336,0.000090,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-05-31,5.000000,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,0.014201,0.010185,0.000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,-100.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000197,0.000141,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,9,0.0,NaN,0.0,0,2026

In [378]:
final_df[final_df['month_date'] == '2026-09-30']['P3M_value'].sum()

31.533487973702506

In [381]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_qcom_trend.csv')

In [380]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_qcom_cp_july_pred.csv')